# Quality Metrics & Dice Score Analysis

1. Compute entropy, voxel volume, file size, and segmentation volume per image.
2. Map coregistered image/segmentation paths.
3. Build modality presence matrix per patient-timepoint.
4. Compute T1-T2 Dice scores at matched timepoints.
5. Compute pairwise and per-study average Dice scores.

In [ ]:
import os
import gc
import glob
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import nibabel as nib
import tifffile
import SimpleITK as sitk
from tqdm.auto import tqdm
from IPython.display import display

tqdm.pandas()

try:
    import ants
    ANTS_AVAILABLE = True
except ImportError:
    ANTS_AVAILABLE = False
    print('Warning: ANTsPy not available. antspy_spacing will be NaN.')

In [ ]:
# ── Configuration ──
meta_csvs = sorted(glob.glob('meta_df_*.csv'))
if not meta_csvs:
    raise FileNotFoundError('No meta_df_*.csv found. Run 01_data_ingestion.ipynb first.')
META_CSV = meta_csvs[-1]
print(f'Using: {META_CSV}')

# Entropy settings
APPROXIMATE_ENTROPY = True
MAX_ENTROPY_SAMPLES = 1_000_000
BINS = 256

# Output CSV for cached results
CSV_PATH = 'meta_df_entropy.csv'

# Coregistered folder name (replaces 'raw_studies' in paths)
COREGISTERED_FOLDER = 'df_coregistered_02_09_2026_V6.6/raw_studies'

In [ ]:
# Load metadata and filter
meta_df = pd.read_csv(META_CSV)
meta_df = meta_df[
    ~meta_df['Study Type'].str.contains(r'T1 pre|T2 Thick', case=False, na=False)
].dropna(subset='image path')
print(f'Loaded {len(meta_df)} rows from {META_CSV}')

## 1. Helper Functions

In [ ]:
def _read_array(path):
    """Read image data from NIfTI, TIFF, or npy file."""
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.nii', '.nii.gz') or path.endswith('.nii.gz'):
        try:
            return nib.load(path).get_fdata(dtype=np.float32)
        except Exception:
            pass
    if tifffile is not None and ext in ('.tif', '.tiff'):
        try:
            with tifffile.TiffFile(path) as tif:
                return tif.asarray(memmap=True)
        except Exception:
            try:
                return tifffile.imread(path)
            except Exception:
                pass
    try:
        img = sitk.ReadImage(path)
        return sitk.GetArrayFromImage(img)
    except Exception:
        pass
    if ext == '.npy':
        return np.load(path, mmap_mode='r')
    raise RuntimeError(f'Could not read {path}')


def compute_entropy(arr, bins=BINS, approximate=APPROXIMATE_ENTROPY, max_samples=MAX_ENTROPY_SAMPLES):
    """Shannon entropy (base 2) via histogram."""
    try:
        flat = arr.ravel()
        n = flat.size
        if n == 0:
            return 0.0
        if approximate and n > max_samples:
            step = int(np.ceil(n / max_samples))
            sample = flat[::step]
        else:
            sample = flat
        if np.issubdtype(sample.dtype, np.floating):
            mask = ~np.isnan(sample)
            if not np.any(mask):
                return 0.0
            sample = sample[mask]
            mn, mx = float(np.min(sample)), float(np.max(sample))
            if np.isclose(mx, mn):
                return 0.0
            counts, _ = np.histogram(sample, bins=bins, range=(mn, mx))
        else:
            mn, mx = int(np.nanmin(sample)), int(np.nanmax(sample))
            if np.isclose(mx, mn):
                return 0.0
            if mx - mn + 1 <= bins:
                counts, _ = np.histogram(sample, bins=(mx - mn + 1), range=(mn - 0.5, mx + 0.5))
            else:
                counts, _ = np.histogram(sample, bins=bins, range=(mn, mx))
        probs = counts.astype(np.float64)
        s = probs.sum()
        if s == 0:
            return 0.0
        probs /= s
        nz = probs > 0
        return float(-np.sum(probs[nz] * np.log2(probs[nz])))
    except Exception:
        return float('nan')


def get_voxel_dims_from_file(path):
    """Extract (dx, dy, dz) and unit from file headers."""
    if path.endswith('.nii.gz') or path.endswith('.nii'):
        try:
            img = nib.load(path)
            zooms = img.header.get_zooms()
            unit = 'mm'
            try:
                xyzt_units = img.header.get_xyzt_units()
                if xyzt_units[0]:
                    unit = xyzt_units[0]
            except Exception:
                pass
            if len(zooms) >= 3:
                return tuple(map(float, zooms[:3])), unit
            elif len(zooms) == 2:
                return (float(zooms[0]), float(zooms[1]), 1.0), unit
        except Exception:
            pass
    try:
        img = sitk.ReadImage(path)
        spacing = img.GetSpacing()
        if len(spacing) >= 3:
            return tuple(map(float, spacing[:3])), 'mm'
        elif len(spacing) == 2:
            return (float(spacing[0]), float(spacing[1]), 1.0), 'mm'
    except Exception:
        pass
    return None, None


def get_antspy_spacing(path):
    """Extract spacing via ANTsPy."""
    if not ANTS_AVAILABLE or not isinstance(path, str) or not os.path.exists(path):
        return None
    try:
        img = ants.image_read(path)
        return tuple(float(s) for s in img.spacing)
    except Exception:
        return None


def get_volume_unit(voxel_unit):
    """Convert linear unit to cubic unit string."""
    if not voxel_unit or pd.isna(voxel_unit):
        return None
    unit_map = {'mm': 'mm\u00b3', 'um': 'um\u00b3', 'cm': 'cm\u00b3', 'pixels': 'pixels\u00b3'}
    return unit_map.get(str(voxel_unit).strip().lower(), f'{voxel_unit}\u00b3')


def compute_segmentation_volume(seg_path, voxel_volume):
    """Volume = non-zero voxels * voxel_volume."""
    try:
        if not isinstance(seg_path, str) or not os.path.exists(seg_path) or np.isnan(voxel_volume):
            return np.nan
        seg_array = _read_array(seg_path)
        return float(np.count_nonzero(seg_array) * voxel_volume)
    except Exception:
        return np.nan


def find_coregistered_path(original_path, coregistered_folder):
    """Replace 'raw_studies' in path with coregistered folder and add suffix."""
    if not isinstance(original_path, str) or pd.isna(original_path):
        return np.nan
    if 'raw_studies' not in original_path.lower():
        return np.nan
    idx = original_path.lower().find('raw_studies')
    coreg_path = original_path[:idx] + coregistered_folder + original_path[idx + len('raw_studies'):]
    coreg_path = coreg_path.replace('.nii.gz', '_coregistered.nii.gz')
    if not os.path.exists(coreg_path):
        return np.nan
    return coreg_path

## 2. Compute Metrics

In [ ]:
def process_row(row):
    """Compute all metrics for a single row."""
    path = row.get('image path')
    if not isinstance(path, str) or not os.path.exists(path):
        return pd.Series({
            'entropy': np.nan, 'voxel_volume': np.nan, 'voxel_unit': None,
            'volume_unit': None, 'file_size_mb': np.nan,
            'coregistered_image': np.nan, 'coregistered_segmentation': np.nan,
            'segmentation_volume': np.nan, 'antspy_spacing': None
        })

    # File size
    try:
        file_size_mb = os.path.getsize(path) / (1024 * 1024)
    except Exception:
        file_size_mb = np.nan

    # Coregistered paths
    coreg_image = find_coregistered_path(path, COREGISTERED_FOLDER)
    seg_path = row.get('segmentation path')
    coreg_seg = find_coregistered_path(seg_path, COREGISTERED_FOLDER) if pd.notna(seg_path) else np.nan

    # ANTsPy spacing
    antspy_sp = get_antspy_spacing(path)

    # Voxel dimensions
    dims, voxel_unit = get_voxel_dims_from_file(path)
    voxel_volume = np.nan
    volume_unit = None
    if dims is not None:
        try:
            voxel_volume = float(np.prod(dims))
            volume_unit = get_volume_unit(voxel_unit)
        except Exception:
            pass

    # Entropy
    try:
        arr = _read_array(path)
        entropy = compute_entropy(arr)
    except Exception:
        entropy = float('nan')

    # Segmentation volume
    seg_vol = np.nan
    if pd.notna(seg_path):
        seg_vol = compute_segmentation_volume(seg_path, voxel_volume)

    return pd.Series({
        'entropy': entropy, 'voxel_volume': voxel_volume, 'voxel_unit': voxel_unit,
        'volume_unit': volume_unit, 'file_size_mb': file_size_mb,
        'coregistered_image': coreg_image, 'coregistered_segmentation': coreg_seg,
        'segmentation_volume': seg_vol, 'antspy_spacing': antspy_sp
    })

In [ ]:
# Compute or load cached results
if os.path.exists(CSV_PATH):
    try:
        meta_df = pd.read_csv(CSV_PATH)
        print(f'Loaded existing {CSV_PATH}; skipping computation.')
    except Exception as e:
        print(f'Error reading {CSV_PATH}: {e}. Recomputing...')
        results = meta_df.progress_apply(process_row, axis=1)
        metric_cols = ['entropy', 'voxel_volume', 'voxel_unit', 'volume_unit',
                       'file_size_mb', 'coregistered_image', 'coregistered_segmentation',
                       'segmentation_volume', 'antspy_spacing']
        meta_df[metric_cols] = results
        meta_df.to_csv(CSV_PATH, index=False)
else:
    print(f'Computing metrics for {len(meta_df)} rows...')
    results = meta_df.progress_apply(process_row, axis=1)
    metric_cols = ['entropy', 'voxel_volume', 'voxel_unit', 'volume_unit',
                   'file_size_mb', 'coregistered_image', 'coregistered_segmentation',
                   'segmentation_volume', 'antspy_spacing']
    meta_df[metric_cols] = results
    meta_df.to_csv(CSV_PATH, index=False)
    print(f'Saved results to {CSV_PATH}')

## 3. Modality Presence Matrix

In [ ]:
# Boolean columns: which modalities exist per patient-timepoint
modality_presence = (
    meta_df.groupby('Patient_MRI_Days Tracker')['Study Type']
    .apply(lambda x: pd.Series(True, index=x.unique()))
    .unstack(fill_value=False)
    .reset_index()
)
modality_presence.columns = ['Patient_MRI_Days Tracker'] + [
    f'has_{col}' for col in modality_presence.columns[1:]
]

# Which modalities have segmentations
seg_col = 'segmentation path' if 'segmentation path' in meta_df.columns else 'original_seg_path'
modality_with_seg = (
    meta_df[meta_df[seg_col].notna()]
    .groupby('Patient_MRI_Days Tracker')['Study Type']
    .apply(lambda x: pd.Series(True, index=x.unique()))
    .unstack(fill_value=False)
    .reset_index()
)
modality_with_seg.columns = ['Patient_MRI_Days Tracker'] + [
    f'has_seg_{col}' for col in modality_with_seg.columns[1:]
]

tracker_modality_df = modality_presence.merge(
    modality_with_seg, on='Patient_MRI_Days Tracker', how='left'
).fillna(False).infer_objects(copy=False)

print('Patient Timepoints with Modality Presence and Segmentation Status:')
display(tracker_modality_df.head(20))

In [ ]:
# Filter for timepoints with BOTH t1+_thin and t2_thin segmentations
both_cols = ['has_seg_t1+_thin', 'has_seg_t2_thin']
available_cols = [c for c in both_cols if c in tracker_modality_df.columns]

if len(available_cols) == 2:
    trackers_with_both = tracker_modality_df[
        (tracker_modality_df['has_seg_t1+_thin'] == True) &
        (tracker_modality_df['has_seg_t2_thin'] == True)
    ]
    print(f'Trackers with BOTH t1+_thin AND t2_thin segmentations: {len(trackers_with_both)}')
    display(trackers_with_both)
else:
    print(f'Available seg columns: {[c for c in tracker_modality_df.columns if c.startswith("has_seg_")]}')
    print('Adjust filter columns as needed.')

## 4. T1 vs T2 Dice Score Comparison

In [ ]:
def calculate_dice_coefficient(seg1_path, seg2_path):
    """Dice coefficient between two binary segmentation masks."""
    try:
        seg1 = (nib.load(seg1_path).get_fdata() > 0).astype(np.uint8)
        seg2 = (nib.load(seg2_path).get_fdata() > 0).astype(np.uint8)

        intersection = np.sum(seg1 * seg2)
        sum_volumes = np.sum(seg1) + np.sum(seg2)

        if sum_volumes == 0:
            return np.nan
        return 2.0 * intersection / sum_volumes
    except Exception as e:
        print(f'Error calculating Dice: {e}')
        return np.nan


def create_dice_comparison(df):
    """Compare T1 vs T2 coregistered segmentations at each timepoint."""
    results = []
    for tracker_id, group in df.groupby('Patient_MRI_Days Tracker'):
        t1_scans = group[group['Study Type'].str.contains('t1', case=False, na=False)]
        t2_scans = group[group['Study Type'].str.contains('t2', case=False, na=False)]

        for _, t1 in t1_scans.iterrows():
            for _, t2 in t2_scans.iterrows():
                if pd.notna(t1['coregistered_segmentation']) and pd.notna(t2['coregistered_segmentation']):
                    dice = calculate_dice_coefficient(
                        t1['coregistered_segmentation'],
                        t2['coregistered_segmentation']
                    )
                    results.append({
                        'Patient_MRI_Days Tracker': tracker_id,
                        't1_thin': t1['coregistered_image'],
                        't2_thin': t2['coregistered_image'],
                        't1_segmentation': t1['coregistered_segmentation'],
                        't2_segmentation': t2['coregistered_segmentation'],
                        'dice_score': dice
                    })
    return pd.DataFrame(results)


dice_df = create_dice_comparison(meta_df)
if dice_df.empty:
    print('No T1-T2 comparisons found (no coregistered segmentation pairs).')
else:
    display(dice_df.head())
    print(f'Patient timepoints with both T1 and T2 segmentations: {dice_df["Patient_MRI_Days Tracker"].nunique()}')
    print(f'Total T1-T2 comparisons: {len(dice_df)}')

## 5. Pairwise Dice Scores Across All Studies per Patient

In [ ]:
def create_pairwise_dice(df):
    """Pairwise Dice scores for all coregistered segmentations within each patient."""
    results = []

    if 'Patient_ID' not in df.columns:
        df = df.copy()
        df['Patient_ID'] = df['Patient_MRI_Days Tracker'].str.split('_').str[0]

    grouped = df.groupby('Patient_ID')

    # Count total comparisons for progress bar
    total = 0
    for _, group in grouped:
        n = len(group[pd.notna(group['coregistered_segmentation'])])
        if n >= 2:
            total += n * (n - 1) // 2

    print(f'Processing {len(grouped)} patients with {total} total pairwise comparisons...')
    pbar = tqdm(total=total, desc='Calculating Dice scores', unit='comparison')

    for patient_id, group in grouped:
        studies = group[pd.notna(group['coregistered_segmentation'])].copy()
        n = len(studies)
        if n < 2:
            continue

        for i in range(n):
            for j in range(i + 1, n):
                s1 = studies.iloc[i]
                s2 = studies.iloc[j]
                dice = calculate_dice_coefficient(
                    s1['coregistered_segmentation'],
                    s2['coregistered_segmentation']
                )
                results.append({
                    'Patient_MRI_Days Tracker': patient_id,
                    'study1_filename': s1['coregistered_image'],
                    'study2_filename': s2['coregistered_image'],
                    'study1_type': s1.get('Study Type', 'Unknown'),
                    'study2_type': s2.get('Study Type', 'Unknown'),
                    'study1_segmentation': s1['coregistered_segmentation'],
                    'study2_segmentation': s2['coregistered_segmentation'],
                    'dice_score': dice
                })
                pbar.update(1)

    pbar.close()
    return pd.DataFrame(results)


pairwise_dice_df = create_pairwise_dice(meta_df)
if pairwise_dice_df.empty:
    print('No pairwise comparisons found (no coregistered segmentation pairs).')
else:
    display(pairwise_dice_df.head(10))
    print(f'\nTotal pairwise comparisons: {len(pairwise_dice_df)}')
    print(f'Unique patients with comparisons: {pairwise_dice_df["Patient_MRI_Days Tracker"].nunique()}')
    print(f'\nDice score statistics:')
    print(pairwise_dice_df['dice_score'].describe())

In [ ]:
if not pairwise_dice_df.empty:
    pairwise_dice_df.to_csv('pairwise_dice_df.csv', index=False)
    print('Saved pairwise_dice_df.csv')
else:
    print('Skipped — no pairwise Dice data to save.')

## 6. Average Dice Score per Study

In [ ]:
def create_average_dice_per_study(pairwise_df):
    """Calculate average Dice score for each study across all its comparisons."""
    all_studies = pd.concat([
        pairwise_df[['Patient_MRI_Days Tracker', 'study1_filename', 'study1_type', 'study1_segmentation']]
            .rename(columns={'study1_filename': 'filename', 'study1_type': 'study_type', 'study1_segmentation': 'segmentation'}),
        pairwise_df[['Patient_MRI_Days Tracker', 'study2_filename', 'study2_type', 'study2_segmentation']]
            .rename(columns={'study2_filename': 'filename', 'study2_type': 'study_type', 'study2_segmentation': 'segmentation'})
    ]).drop_duplicates()

    print(f'Calculating average Dice scores for {len(all_studies)} unique studies...')
    results = []

    for _, study_row in tqdm(all_studies.iterrows(), total=len(all_studies), desc='Processing studies'):
        tracker_id = study_row['Patient_MRI_Days Tracker']
        filename = study_row['filename']

        as_s1 = pairwise_df[
            (pairwise_df['Patient_MRI_Days Tracker'] == tracker_id) &
            (pairwise_df['study1_filename'] == filename)
        ]['dice_score']
        as_s2 = pairwise_df[
            (pairwise_df['Patient_MRI_Days Tracker'] == tracker_id) &
            (pairwise_df['study2_filename'] == filename)
        ]['dice_score']

        all_dice = pd.concat([as_s1, as_s2]).dropna()
        if len(all_dice) > 0:
            results.append({
                'Patient_MRI_Days Tracker': tracker_id,
                'study_filename': filename,
                'study_type': study_row['study_type'],
                'segmentation': study_row['segmentation'],
                'n_comparisons': len(all_dice),
                'mean_dice': all_dice.mean(),
                'std_dice': all_dice.std(),
                'min_dice': all_dice.min(),
                'max_dice': all_dice.max(),
                'median_dice': all_dice.median()
            })

    avg_df = pd.DataFrame(results).sort_values('mean_dice', ascending=False).reset_index(drop=True)
    return avg_df


if not pairwise_dice_df.empty:
    avg_dice_df = create_average_dice_per_study(pairwise_dice_df)

    print(f'\nTotal studies analyzed: {len(avg_dice_df)}')
    print(f'\nMean Dice statistics:')
    print(avg_dice_df['mean_dice'].describe())

    print(f'\nAverage Dice by Study Type:')
    print(avg_dice_df.groupby('study_type').agg(
        mean_dice=('mean_dice', 'mean'),
        std_dice=('mean_dice', 'std'),
        count=('mean_dice', 'count')
    ).round(4))

    print(f'\nTop 10 Studies (Highest Dice):')
    display(avg_dice_df.head(10)[['Patient_MRI_Days Tracker', 'study_filename', 'study_type', 'n_comparisons', 'mean_dice']])

    print(f'\nBottom 10 Studies (Lowest Dice):')
    display(avg_dice_df.tail(10)[['Patient_MRI_Days Tracker', 'study_filename', 'study_type', 'n_comparisons', 'mean_dice']])

    avg_dice_df.to_csv('avg_dice_per_study_df.csv', index=False)
    print('Saved avg_dice_per_study_df.csv')
else:
    print('Skipped — no pairwise Dice data available.')